# Data Data Collection for Bitcoin Trading

This notebook collects historical Bitcoin price data from cryptocurrency exchanges.

**Project:** DeepTrade-RL - A Reinforcement Learning Bitcoin Trading Agent  
**Course:** SE4050 Deep Learning

## Objectives:
1. Connect to Binance Testnet API
2. Fetch historical BTC/USDT price data
3. Calculate technical indicators
4. Save processed data for training

---

In [ ]:
# Install required packages (Google Colab)
!pip install -q requests pandas numpy matplotlib ta-lib

# Mount Google Drive to save data
from google.colab import drive
drive.mount('/content/drive')

## Import Libraries

In [ ]:
import sys
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import time

# Set up project path
PROJECT_PATH = "/content/drive/MyDrive/deeptrade-rl"
os.makedirs(PROJECT_PATH, exist_ok=True)
os.makedirs(f"{PROJECT_PATH}/data", exist_ok=True)

print(f"Project directory: {PROJECT_PATH}")
print(f"Python version: {sys.version}")

## 1. Fetch Historical Data from Binance

We'll use the Binance public API to fetch historical OHLCV (Open, High, Low, Close, Volume) data for BTC/USDT.

In [ ]:
def fetch_binance_data(symbol="BTCUSDT", interval="1h", limit=1000, start_time=None):
    """
    Fetch historical candlestick data from Binance.
    
    Args:
        symbol: Trading pair (e.g., BTCUSDT)
        interval: Kline interval (1m, 5m, 15m, 1h, 4h, 1d)
        limit: Number of data points (max 1000)
        start_time: Start timestamp in milliseconds
        
    Returns:
        DataFrame with OHLCV data
    """
    base_url = "https://api.binance.com/api/v3/klines"
    
    params = {
        "symbol": symbol,
        "interval": interval,
        "limit": limit
    }
    
    if start_time:
        params["startTime"] = start_time
    
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()
        
        # Convert to DataFrame
        df = pd.DataFrame(data, columns=[
            "timestamp", "open", "high", "low", "close", "volume",
            "close_time", "quote_volume", "trades", "taker_buy_base",
            "taker_buy_quote", "ignore"
        ])
        
        # Convert types
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        for col in ["open", "high", "low", "close", "volume"]:
            df[col] = df[col].astype(float)
        
        return df[["timestamp", "open", "high", "low", "close", "volume"]]
    
    except Exception as e:
        print(f"Error fetching data: {e}")
        return pd.DataFrame()

# Test fetch
print("Fetching BTC/USDT hourly data...")
df_sample = fetch_binance_data("BTCUSDT", interval="1h", limit=100)
print(f"Fetched {len(df_sample)} data points")
print(df_sample.head())

## 2. Fetch Extended Historical Data

For training, we need multiple months of data. We'll fetch data in batches.

In [ ]:
def fetch_extended_data(symbol="BTCUSDT", interval="1h", days=180):
    """
    Fetch extended historical data by making multiple API calls.
    
    Args:
        symbol: Trading pair
        interval: Kline interval
        days: Number of days of historical data
        
    Returns:
        Combined DataFrame
    """
    all_data = []
    
    # Calculate time range
    end_time = datetime.now()
    start_time = end_time - timedelta(days=days)
    
    current_time = start_time
    batch_count = 0
    
    print(f"Fetching {days} days of {interval} data for {symbol}...")
    
    while current_time < end_time:
        start_ms = int(current_time.timestamp() * 1000)
        
        df_batch = fetch_binance_data(
            symbol=symbol,
            interval=interval,
            limit=1000,
            start_time=start_ms
        )
        
        if df_batch.empty:
            break
        
        all_data.append(df_batch)
        
        # Update current time to last timestamp + 1 hour
        current_time = df_batch['timestamp'].max() + timedelta(hours=1)
        batch_count += 1
        
        print(f"  Batch {batch_count}: Fetched {len(df_batch)} rows (up to {current_time})")
        
        # Rate limiting
        time.sleep(0.5)
    
    # Combine all data
    if all_data:
        df_combined = pd.concat(all_data, ignore_index=True)
        df_combined = df_combined.drop_duplicates(subset=['timestamp']).sort_values('timestamp').reset_index(drop=True)
        print(f"\nTotal data points: {len(df_combined)}")
        return df_combined
    else:
        return pd.DataFrame()

# Fetch 6 months of hourly data (approximately 4,380 data points)
df_btc = fetch_extended_data("BTCUSDT", interval="1h", days=180)

print(f"\nChart Data Summary:")
print(f"  Date range: {df_btc['timestamp'].min()} to {df_btc['timestamp'].max()}")
print(f"  Total rows: {len(df_btc)}")
print(f"  Price range: ${df_btc['close'].min():,.2f} - ${df_btc['close'].max():,.2f}")

## 3. Calculate Technical Indicators

Add RSI, EMA, MACD, Bollinger Bands, and other indicators.

In [ ]:
def calculate_rsi(prices, period=14):
    """Calculate Relative Strength Index."""
    delta = prices.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)
    
    avg_gain = gain.rolling(window=period, min_periods=period).mean()
    avg_loss = loss.rolling(window=period, min_periods=period).mean()
    
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

def calculate_ema(prices, period=20):
    """Calculate Exponential Moving Average."""
    return prices.ewm(span=period, adjust=False).mean()

def calculate_macd(prices, fast=12, slow=26, signal=9):
    """Calculate MACD indicator."""
    ema_fast = calculate_ema(prices, fast)
    ema_slow = calculate_ema(prices, slow)
    macd_line = ema_fast - ema_slow
    signal_line = calculate_ema(macd_line, signal)
    histogram = macd_line - signal_line
    return macd_line, signal_line, histogram

def add_technical_indicators(df):
    """Add all technical indicators to dataframe."""
    df = df.copy()
    
    # RSI
    df['rsi'] = calculate_rsi(df['close'], period=14)
    
    # EMAs
    df['ema_12'] = calculate_ema(df['close'], period=12)
    df['ema_26'] = calculate_ema(df['close'], period=26)
    df['ema_50'] = calculate_ema(df['close'], period=50)
    
    # SMAs
    df['sma_20'] = df['close'].rolling(window=20).mean()
    df['sma_50'] = df['close'].rolling(window=50).mean()
    
    # MACD
    macd, signal, histogram = calculate_macd(df['close'])
    df['macd'] = macd
    df['macd_signal'] = signal
    df['macd_histogram'] = histogram
    
    # Bollinger Bands
    sma_20 = df['sma_20']
    std_20 = df['close'].rolling(window=20).std()
    df['bb_upper'] = sma_20 + (std_20 * 2)
    df['bb_lower'] = sma_20 - (std_20 * 2)
    df['bb_width'] = (df['bb_upper'] - df['bb_lower']) / sma_20
    
    # Price momentum
    df['momentum'] = df['close'].pct_change(periods=10) * 100
    
    # Volume indicators
    df['volume_sma'] = df['volume'].rolling(window=20).mean()
    df['volume_ratio'] = df['volume'] / df['volume_sma']
    
    return df

# Add indicators
print("Calculating technical indicators...")
df_btc_with_indicators = add_technical_indicators(df_btc)

# Remove NaN values from indicator calculations
df_btc_with_indicators = df_btc_with_indicators.dropna().reset_index(drop=True)

print(f"Data after adding indicators: {len(df_btc_with_indicators)} rows")
print(f"Columns: {list(df_btc_with_indicators.columns)}")

## 4. Visualize Data and Indicators

In [ ]:
# Plot price and indicators
fig, axes = plt.subplots(4, 1, figsize=(15, 12))

# Price with EMAs
axes[0].plot(df_btc_with_indicators['timestamp'], df_btc_with_indicators['close'], label='BTC Price', linewidth=1)
axes[0].plot(df_btc_with_indicators['timestamp'], df_btc_with_indicators['ema_12'], label='EMA 12', alpha=0.7)
axes[0].plot(df_btc_with_indicators['timestamp'], df_btc_with_indicators['ema_50'], label='EMA 50', alpha=0.7)
axes[0].set_ylabel('Price (USD)')
axes[0].set_title('BTC/USDT Price with EMAs')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# RSI
axes[1].plot(df_btc_with_indicators['timestamp'], df_btc_with_indicators['rsi'], label='RSI', color='purple')
axes[1].axhline(y=70, color='r', linestyle='--', alpha=0.5, label='Overbought (70)')
axes[1].axhline(y=30, color='g', linestyle='--', alpha=0.5, label='Oversold (30)')
axes[1].set_ylabel('RSI')
axes[1].set_title('Relative Strength Index (RSI)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# MACD
axes[2].plot(df_btc_with_indicators['timestamp'], df_btc_with_indicators['macd'], label='MACD', color='blue')
axes[2].plot(df_btc_with_indicators['timestamp'], df_btc_with_indicators['macd_signal'], label='Signal', color='red')
axes[2].bar(df_btc_with_indicators['timestamp'], df_btc_with_indicators['macd_histogram'], label='Histogram', alpha=0.3)
axes[2].set_ylabel('MACD')
axes[2].set_title('MACD Indicator')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# Volume
axes[3].bar(df_btc_with_indicators['timestamp'], df_btc_with_indicators['volume'], label='Volume', alpha=0.6)
axes[3].set_ylabel('Volume')
axes[3].set_xlabel('Date')
axes[3].set_title('Trading Volume')
axes[3].legend()
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{PROJECT_PATH}/data/btc_indicators.png", dpi=150, bbox_inches='tight')
plt.show()

print("Visualization saved!")

## 5. Save Processed Data

In [ ]:
# Save to CSV
output_file = f"{PROJECT_PATH}/data/cached_market_data.csv"
df_btc_with_indicators.to_csv(output_file, index=False)

print(f"Data saved to: {output_file}")
print(f"   Total rows: {len(df_btc_with_indicators)}")
print(f"   File size: {os.path.getsize(output_file) / 1024:.2f} KB")

# Display summary statistics
print("\nData Summary Statistics:")
print(df_btc_with_indicators[['close', 'volume', 'rsi', 'macd']].describe())

## Data Collection Complete!

**Summary:**
- Fetched historical BTC/USDT data from Binance
- Calculated technical indicators (RSI, EMA, MACD, etc.)
- Visualized price trends and indicators
- Saved processed data for training

**Next Steps:**
- Notebook 02: Test trading environment with this data
- Notebook 03: Train DDQN agent
- Notebook 04: Train PPO agent

---

**Note:** This data is for educational purposes and paper trading only!